# 第三课：异常处理

本课目标：

- 理解异常是什么，以及异常和普通返回值的区别；
- 掌握 `try`、`except`、`else`、`finally`；
- 只捕获自己能够处理的具体异常；
- 使用 `raise` 主动报告无效数据；
- 为函数选择合适的异常类型。

请从上到下逐格运行，不要直接 Run All。

## 1. 什么是异常

程序遇到无法正常继续的情况时，会创建并抛出异常。例如把无法转换的字符串传给 `int()`：

In [ ]:
# 请先运行并观察报错的最后一行。
int("hello")

最后一行会告诉我们异常类型是 `ValueError`，并说明字符串不能转换为整数。报错信息不是敌人，而是定位问题的重要线索。

## 2. 捕获能够处理的具体异常

`try`中放可能失败的操作，`except`中放出现指定异常后的处理：

In [ ]:
def parse_age(value):
    try:
        return int(value)
    except (TypeError, ValueError):
        return None


print(parse_age("20"))
print(parse_age("hello"))
print(parse_age(None))

不要随便写裸 `except:`。它会捕获几乎所有异常，可能把程序错误和用户主动中断也隐藏起来。

也不要为了“程序不报错”而捕获异常。只有当前代码知道怎样恢复、转换或补充信息时，才适合处理异常。

## 3. `else` 与 `finally`

- `else`：`try`没有抛出异常时执行；
- `finally`：无论是否抛出异常，最后都会执行，常用于释放资源。

In [ ]:
def divide(left, right):
    try:
        result = left / right
    except ZeroDivisionError:
        print("除数不能为零")
        return None
    else:
        print("计算成功")
        return result
    finally:
        print("计算结束")


print("结果：", divide(10, 2))
print("---")
print("结果：", divide(10, 0))

## 4. 使用 `raise` 主动报告错误

异常不只由Python自动产生。函数发现调用者给的数据不符合约定时，也可以主动抛出异常：

In [ ]:
def set_temperature(value):
    if not isinstance(value, (int, float)):
        raise TypeError("温度必须是数字")
    if value < -273.15:
        raise ValueError("温度不能低于绝对零度")
    return value


print(set_temperature(25))

常用选择：

- 类型不对：`TypeError`；
- 类型正确但取值无效：`ValueError`；
- 字典缺少必需键：`KeyError`；
- 文件不存在：通常让 `FileNotFoundError` 继续向上传递。

异常信息应说明违反了什么规则，而不是只写“出错了”。

## 5. 动手作业

打开同目录的 `config_loader.py`，完成 `load_config()`。文件的文档字符串中列出了全部要求。

完成并保存后，重新启动Kernel或运行下面单元格中的 `reload()`，再执行测试。不要修改测试来迁就自己的代码。

In [ ]:
import importlib
import config_loader

importlib.reload(config_loader)
load_config = config_loader.load_config

assert load_config({"host": "127.0.0.1"}) == {
    "host": "127.0.0.1",
    "port": 8000,
}
assert load_config({"host": "0.0.0.0", "port": 8080}) == {
    "host": "0.0.0.0",
    "port": 8080,
}
print("合法配置测试通过")

In [ ]:
def assert_raises(expected_exception, function, argument):
    try:
        function(argument)
    except expected_exception:
        return
    except Exception as error:
        raise AssertionError(
            f"期望 {expected_exception.__name__}，实际得到 {type(error).__name__}"
        ) from error
    raise AssertionError(f"期望抛出 {expected_exception.__name__}，但没有异常")


assert_raises(TypeError, load_config, None)
assert_raises(KeyError, load_config, {})
assert_raises(ValueError, load_config, {"host": ""})
assert_raises(TypeError, load_config, {"host": "localhost", "port": "8000"})
assert_raises(ValueError, load_config, {"host": "localhost", "port": 0})
assert_raises(ValueError, load_config, {"host": "localhost", "port": 70000})
print("异常类型测试通过")

## 6. 理解题

完成作业后回答：

1. 为什么通常不推荐写裸 `except:`？
2. `TypeError`和`ValueError`分别适合报告什么问题？
3. `finally`中的代码会在什么时候执行？
4. 为什么`load_config()`不应该把所有异常捕获后统一返回`None`？